# Hurricane Melissa Exposure of Mangroves with >10% NDVI Damage

This is a new notebook that keeps existing notebooks/files unchanged.

It uses the same substantial-damage definition as your NDVI notebook:
- Relative NDVI change: `(after - before) / before < -0.10`
- Eligibility filter: `before >= 0.20`

Analyses:
1. Damage rate inside vs outside hurricane wind swath
2. Damage rate by distance to track (bins)
3. Damage rate by nearest-track intensity / pressure
4. Spatial concentration of damaged mangroves by side/quadrant relative to track


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.ops import unary_union

plt.style.use('default')
pd.set_option('display.max_columns', 150)


In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'dphil_papers').exists():
            return p
    raise FileNotFoundError(f'Could not find project root from {start}')

ROOT = find_project_root(Path.cwd())

# NDVI + mangroves
ndvi_before_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif'
ndvi_after_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/ndvi/HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif'
fn_mangroves_path = ROOT / 'dphil_papers/dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'

# Hurricane Melissa track
track_dir = ROOT / 'dphil_papers/dphil_paper_3/inputs/hurricane_melissa_track_noaa/al132025_best_track'
line_path = track_dir / 'AL132025_lin.shp'
pts_path = track_dir / 'AL132025_pts.shp'
radii_path = track_dir / 'AL132025_radii.shp'
windswath_path = track_dir / 'AL132025_windswath.shp'

jamaica_boundary_path = ROOT / 'dphil_papers/dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

# Keep aligned with NDVI notebook's substantial-change setup
REL_DAMAGE_THRESHOLD = -0.10   # >10% damage means relative change < -0.10
REL_BASELINE_MIN = 0.20        # only evaluate % change where baseline NDVI is high enough
DISTANCE_BINS_KM = [0, 25, 50, 75, 100, 150, 200, 300, 500, 1000]
TRACK_AOI_BUFFER_KM = 200      # clip hurricane layers to Jamaica + this buffer

output_dir = ROOT / 'dphil_papers/dphil_paper_3/results/hurricane_melissa_track_analysis'
output_dir.mkdir(parents=True, exist_ok=True)
SAVE_OUTPUTS = False

for p in [ndvi_before_path, ndvi_after_path, fn_mangroves_path, line_path, pts_path, radii_path, windswath_path, jamaica_boundary_path]:
    print(p.name, 'exists ->', p.exists())
print('Output dir:', output_dir)


In [ ]:
def maybe_save(fig, out_path: Path, dpi=300):
    if SAVE_OUTPUTS:
        fig.savefig(out_path, dpi=dpi)
        print('Saved:', out_path)
    else:
        print('PNG export skipped (SAVE_OUTPUTS=False):', out_path.name)

def coerce_track_to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = gdf.copy()
    b = g.total_bounds
    looks_like_lonlat = (abs(b[0]) <= 180 and abs(b[2]) <= 180 and abs(b[1]) <= 90 and abs(b[3]) <= 90)
    if looks_like_lonlat:
        return g.set_crs(epsg=4326, allow_override=True)
    return g.to_crs(4326)

def parse_track_timestamp(df: pd.DataFrame) -> pd.Series:
    req = {'YEAR', 'MONTH', 'DAY', 'HHMM'}
    if not req.issubset(df.columns):
        return pd.Series(pd.NaT, index=df.index)

    y = pd.to_numeric(df['YEAR'], errors='coerce').astype('Int64')
    m = pd.to_numeric(df['MONTH'], errors='coerce').astype('Int64')
    d = pd.to_numeric(df['DAY'], errors='coerce').astype('Int64')
    hhmm = pd.to_numeric(df['HHMM'], errors='coerce')
    hh = (hhmm // 100).fillna(0).astype('Int64')
    mm = (hhmm % 100).fillna(0).astype('Int64')

    return pd.to_datetime(
        {'year': y, 'month': m, 'day': d, 'hour': hh, 'minute': mm},
        errors='coerce',
        utc=True,
    )

def quadrant_from_dxdy(dx, dy):
    if dx >= 0 and dy >= 0:
        return 'NE'
    if dx >= 0 and dy < 0:
        return 'SE'
    if dx < 0 and dy >= 0:
        return 'NW'
    return 'SW'


In [ ]:
# Build eligible mangrove pixel table with relative NDVI change
fn = gpd.read_file(fn_mangroves_path).to_crs(3448)
fn = fn[fn.geometry.notnull() & ~fn.geometry.is_empty].copy()

with rasterio.open(ndvi_before_path) as src_b, rasterio.open(ndvi_after_path) as src_a:
    if src_b.crs != src_a.crs or src_b.transform != src_a.transform or src_b.shape != src_a.shape:
        raise ValueError('Before/after NDVI rasters are not on same grid.')

    arr_before = src_b.read(1)
    arr_after = src_a.read(1)
    transform = src_b.transform

    mangrove_mask = geometry_mask(
        [g for g in fn.geometry if g is not None and not g.is_empty],
        transform=transform,
        out_shape=(src_b.height, src_b.width),
        invert=True,
    )

    valid_before = np.isfinite(arr_before) & (arr_before >= -1.0) & (arr_before <= 1.0)
    valid_after = np.isfinite(arr_after) & (arr_after >= -1.0) & (arr_after <= 1.0)

    if src_b.nodata is not None and np.isfinite(src_b.nodata):
        valid_before &= arr_before != src_b.nodata
    if src_a.nodata is not None and np.isfinite(src_a.nodata):
        valid_after &= arr_after != src_a.nodata

paired_mask = valid_before & valid_after & mangrove_mask
eligible_mask = paired_mask & (arr_before >= REL_BASELINE_MIN)

delta = arr_after - arr_before
rel_change = np.full(arr_before.shape, np.nan, dtype='float32')
rel_change[eligible_mask] = delta[eligible_mask] / arr_before[eligible_mask]
damaged_mask = eligible_mask & (rel_change < REL_DAMAGE_THRESHOLD)

rows, cols = np.where(eligible_mask)
xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')

eligible_pixels = gpd.GeoDataFrame(
    {
        'row': rows,
        'col': cols,
        'ndvi_before': arr_before[eligible_mask],
        'ndvi_after': arr_after[eligible_mask],
        'ndvi_delta': delta[eligible_mask],
        'rel_change': rel_change[eligible_mask],
        'damaged_gt10pct': (rel_change[eligible_mask] < REL_DAMAGE_THRESHOLD),
    },
    geometry=gpd.points_from_xy(xs, ys),
    crs=3448,
)

print('Paired mangrove pixels:', f'{int(paired_mask.sum()):,}')
print('Eligible for relative-change analysis (before>=0.20):', f'{len(eligible_pixels):,}')
print('Damaged >10% pixels:', f'{int(eligible_pixels.damaged_gt10pct.sum()):,}')
print('Damage rate among eligible (%):', round(float(100*eligible_pixels.damaged_gt10pct.mean()), 3))


In [ ]:
# Load hurricane layers and project to EPSG:3448 for metric analysis
track_line_full = coerce_track_to_wgs84(gpd.read_file(line_path)).to_crs(3448)
track_pts_full = coerce_track_to_wgs84(gpd.read_file(pts_path)).to_crs(3448)
track_radii_full = coerce_track_to_wgs84(gpd.read_file(radii_path)).to_crs(3448)
track_windswath_full = coerce_track_to_wgs84(gpd.read_file(windswath_path)).to_crs(3448)
jamaica = gpd.read_file(jamaica_boundary_path).to_crs(3448)

# Build Jamaica-focused AOI and clip track layers to avoid far-off ocean segments
jamaica_union = unary_union(jamaica.geometry.tolist())
track_aoi_geom = jamaica_union.buffer(TRACK_AOI_BUFFER_KM * 1000)
track_aoi = gpd.GeoDataFrame(geometry=[track_aoi_geom], crs=3448)

# Clip polygons/lines; subset points by intersection
track_line = gpd.overlay(track_line_full, track_aoi, how='intersection')
track_radii = gpd.overlay(track_radii_full, track_aoi, how='intersection')
track_windswath = gpd.overlay(track_windswath_full, track_aoi, how='intersection')
track_pts = track_pts_full[track_pts_full.intersects(track_aoi_geom)].copy()

# Fallbacks if AOI clipping is too strict
if len(track_line) == 0:
    print('Warning: clipped track_line is empty; falling back to full track_line')
    track_line = track_line_full.copy()
if len(track_pts) == 0:
    print('Warning: clipped track_pts is empty; falling back to full track_pts')
    track_pts = track_pts_full.copy()
if len(track_windswath) == 0:
    print('Warning: clipped windswath is empty; falling back to full windswath')
    track_windswath = track_windswath_full.copy()

for c in ['INTENSITY', 'MSLP']:
    if c in track_pts.columns:
        track_pts[c] = pd.to_numeric(track_pts[c], errors='coerce')
track_pts['timestamp'] = parse_track_timestamp(track_pts)

print('AOI buffer (km):', TRACK_AOI_BUFFER_KM)
print('Track points full/clipped:', len(track_pts_full), '/', len(track_pts))
print('Track line full/clipped:', len(track_line_full), '/', len(track_line))
print('Track windswath full/clipped:', len(track_windswath_full), '/', len(track_windswath))


In [ ]:
# Step 1: Inside vs outside wind swath (damage rate comparison)
windswath_union = unary_union(track_windswath.geometry.tolist()) if len(track_windswath) else None
eligible_pixels['inside_windswath'] = False if (windswath_union is None or windswath_union.is_empty) else eligible_pixels.geometry.within(windswath_union)

step1 = (
    eligible_pixels.groupby('inside_windswath')
    .agg(
        n_eligible=('damaged_gt10pct', 'size'),
        n_damaged_gt10pct=('damaged_gt10pct', 'sum'),
        damage_rate_pct=('damaged_gt10pct', lambda s: 100*np.mean(s)),
        median_rel_change=('rel_change', 'median'),
        median_delta=('ndvi_delta', 'median'),
    )
    .reset_index()
)
step1['inside_windswath'] = step1['inside_windswath'].map({True: 'Inside wind swath', False: 'Outside wind swath'})
step1 = step1.round(4)
step1


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), constrained_layout=True)
ax.bar(step1['inside_windswath'], step1['damage_rate_pct'], color=['#d7301f', '#4575b4'])
ax.set_ylabel('Damage rate (% of eligible mangrove pixels)')
ax.set_title('Step 1: >10% Damage Rate Inside vs Outside Wind Swath')
for i, v in enumerate(step1['damage_rate_pct']):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom')
maybe_save(fig, output_dir / 'gt10_step1_damage_rate_inside_outside_swath.png')
plt.show()


In [ ]:
# Step 2: Damage rate by distance to track
line_union = unary_union(track_line.geometry.tolist())
eligible_pixels['distance_to_track_km'] = eligible_pixels.geometry.distance(line_union) / 1000.0

bins = DISTANCE_BINS_KM + [np.inf]
labels = [f'{bins[i]}-{bins[i+1]} km' if np.isfinite(bins[i+1]) else f'{bins[i]}+ km' for i in range(len(bins)-1)]
eligible_pixels['distance_bin'] = pd.cut(eligible_pixels['distance_to_track_km'], bins=bins, labels=labels, right=False)

step2 = (
    eligible_pixels.groupby('distance_bin', observed=True)
    .agg(
        n_eligible=('damaged_gt10pct', 'size'),
        n_damaged_gt10pct=('damaged_gt10pct', 'sum'),
        damage_rate_pct=('damaged_gt10pct', lambda s: 100*np.mean(s)),
        median_rel_change=('rel_change', 'median'),
    )
    .reset_index()
    .round(4)
)
step2


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
ax.plot(step2['distance_bin'].astype(str), step2['damage_rate_pct'], marker='o', color='#1f78b4')
ax.set_ylabel('Damage rate (% >10% decline)')
ax.set_xlabel('Distance bin from track')
ax.set_title('Step 2: >10% Damage Rate vs Distance to Track')
ax.tick_params(axis='x', rotation=45)
maybe_save(fig, output_dir / 'gt10_step2_damage_rate_by_distance_bins.png')
plt.show()


In [ ]:
# Step 3: Damage rate by nearest-track intensity / pressure
pts_for_join = track_pts[['geometry']].copy()
for c in ['INTENSITY', 'MSLP', 'timestamp']:
    if c in track_pts.columns:
        pts_for_join[c] = track_pts[c]
pts_for_join['track_x'] = track_pts.geometry.x
pts_for_join['track_y'] = track_pts.geometry.y

exp = gpd.sjoin_nearest(
    eligible_pixels,
    pts_for_join,
    how='left',
    distance_col='distance_to_nearest_track_point_m'
)

if 'INTENSITY' in exp.columns and exp['INTENSITY'].notna().sum() > 0:
    exp['intensity_q'] = pd.qcut(exp['INTENSITY'], q=4, duplicates='drop')
    step3_intensity = (
        exp.groupby('intensity_q', observed=True)
        .agg(
            n_eligible=('damaged_gt10pct', 'size'),
            n_damaged_gt10pct=('damaged_gt10pct', 'sum'),
            damage_rate_pct=('damaged_gt10pct', lambda s: 100*np.mean(s)),
            median_rel_change=('rel_change', 'median'),
        )
        .reset_index()
        .round(4)
    )
else:
    step3_intensity = pd.DataFrame()

if 'MSLP' in exp.columns and exp['MSLP'].notna().sum() > 0:
    exp['mslp_q'] = pd.qcut(exp['MSLP'], q=4, duplicates='drop')
    step3_mslp = (
        exp.groupby('mslp_q', observed=True)
        .agg(
            n_eligible=('damaged_gt10pct', 'size'),
            n_damaged_gt10pct=('damaged_gt10pct', 'sum'),
            damage_rate_pct=('damaged_gt10pct', lambda s: 100*np.mean(s)),
            median_rel_change=('rel_change', 'median'),
        )
        .reset_index()
        .round(4)
    )
else:
    step3_mslp = pd.DataFrame()

print('Step 3A - by intensity quantile:')
display(step3_intensity)
print('Step 3B - by MSLP quantile:')
display(step3_mslp)


In [ ]:
# Step 4: Spatial concentration of >10% damaged pixels by side/quadrant
damage = exp[exp['damaged_gt10pct']].copy()

damage['dx_m'] = damage.geometry.x - damage['track_x']
damage['dy_m'] = damage.geometry.y - damage['track_y']
damage['quadrant'] = [quadrant_from_dxdy(dx, dy) for dx, dy in zip(damage['dx_m'], damage['dy_m'])]
damage['side'] = np.where(damage['dx_m'] >= 0, 'East of nearest track point', 'West of nearest track point')

step4_quad = (
    damage.groupby('quadrant', observed=True)
    .agg(n_damaged=('damaged_gt10pct', 'size'))
    .reset_index()
)
step4_quad['pct_of_damaged'] = 100 * step4_quad['n_damaged'] / max(len(damage), 1)
step4_quad = step4_quad.sort_values('quadrant').reset_index(drop=True)

step4_side = (
    damage.groupby('side', observed=True)
    .agg(n_damaged=('damaged_gt10pct', 'size'))
    .reset_index()
)
step4_side['pct_of_damaged'] = 100 * step4_side['n_damaged'] / max(len(damage), 1)

print('Damaged (>10%) mangrove pixels:', f'{len(damage):,}')
print('Quadrant concentration:')
display(step4_quad.round(3))
print('Side concentration:')
display(step4_side.round(3))


In [ ]:
plot_damage = damage if len(damage) <= 100000 else damage.sample(100000, random_state=42)
quad_colors = {'NE': '#1b9e77', 'SE': '#d95f02', 'SW': '#7570b3', 'NW': '#e7298a'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
ax_map, ax_bar = axes

jamaica.boundary.plot(ax=ax_map, color='black', linewidth=0.8, alpha=0.8)
track_line.plot(ax=ax_map, color='#08519c', linewidth=1.6, alpha=0.9)
if len(track_windswath) > 0:
    track_windswath.plot(ax=ax_map, color='#9ecae1', alpha=0.15, edgecolor='none')

for q, col in quad_colors.items():
    sub = plot_damage[plot_damage['quadrant'] == q]
    if len(sub) > 0:
        sub.plot(ax=ax_map, markersize=3, color=col, alpha=0.55, label=q)

ax_map.set_title('Step 4: >10% Damaged Mangroves by Quadrant')
ax_map.set_xlabel('Easting (m, EPSG:3448)')
ax_map.set_ylabel('Northing (m, EPSG:3448)')
ax_map.set_aspect('equal')
ax_map.legend(title='Quadrant', loc='upper right', frameon=True)

ax_bar.bar(step4_quad['quadrant'], step4_quad['pct_of_damaged'], color=[quad_colors.get(q, '#999999') for q in step4_quad['quadrant']])
ax_bar.set_title('Concentration of >10% damaged pixels by quadrant')
ax_bar.set_xlabel('Quadrant')
ax_bar.set_ylabel('% of damaged pixels')
for i, v in enumerate(step4_quad['pct_of_damaged']):
    ax_bar.text(i, v + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

maybe_save(fig, output_dir / 'gt10_step4_damage_quadrant_map_and_bar.png')
plt.show()


In [ ]:
# Overlay map: >10% relative NDVI change (same red/grey/green scheme) with track line
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

# Classify full raster for mangrove paired pixels:
# -1 = >10% decrease, 0 = not substantial / low-baseline, +1 = >10% increase
cls_rel = np.full(arr_before.shape, np.nan, dtype='float32')
cls_rel[paired_mask] = 0.0
cls_rel[eligible_mask & (rel_change < REL_DAMAGE_THRESHOLD)] = -1.0
cls_rel[eligible_mask & (rel_change > abs(REL_DAMAGE_THRESHOLD))] = 1.0

n_dec = int(np.nansum(cls_rel == -1.0))
n_mid = int(np.nansum(cls_rel == 0.0))
n_inc = int(np.nansum(cls_rel == 1.0))
n_tot = n_dec + n_mid + n_inc

# Same scheme as mangrove NDVI notebook
cmap = ListedColormap(['#d73027', '#d9d9d9', '#1a9850'])  # red, grey, green
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

# Raster extent in EPSG:3448
h, w = arr_before.shape
left, bottom, right, top = rasterio.transform.array_bounds(h, w, transform)

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls_rel,
    cmap=cmap,
    norm=norm,
    extent=[left, right, bottom, top],
    origin='upper',
    interpolation='nearest',
)

# Overlay hurricane wind thresholds and track
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw = track_windswath.copy()
    sw['RADII'] = pd.to_numeric(sw['RADII'], errors='coerce')
    threshold_colors = {34.0: '#9ecae1', 50.0: '#4292c6', 64.0: '#08519c'}

    for thr in sorted(sw['RADII'].dropna().unique()):
        subset = sw[sw['RADII'] == thr]
        color = threshold_colors.get(float(thr), '#6baed6')
        if len(subset) > 0:
            subset.boundary.plot(ax=ax, color=color, linewidth=1.0, alpha=0.9)

track_line.plot(ax=ax, color='black', linewidth=1.8, alpha=0.9)
jamaica.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)

ax.set_title(
    f'FN Mangroves: Relative NDVI Change with Hurricane Melissa Track\n'
    f'Red: decrease >10% | Green: increase >10% | Grey: not substantial/low baseline (before < {REL_BASELINE_MIN:.2f})',
    fontsize=11,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI decrease > 10%'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Not substantial / low baseline'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI increase > 10%'),
    Line2D([0], [0], color='black', lw=1.8, label='Melissa track line'),
]

# Add wind threshold legend entries if available
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    for thr, col in [(34.0, '#9ecae1'), (50.0, '#4292c6'), (64.0, '#08519c')]:
        if np.any(pd.to_numeric(track_windswath['RADII'], errors='coerce') == thr):
            legend_handles.append(Line2D([0], [0], color=col, lw=1.2, label=f'Windswath {int(thr)} kt'))

ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95, fontsize=9)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  dec={100*n_dec/max(n_tot,1):.1f}%  other={100*n_mid/max(n_tot,1):.1f}%  inc={100*n_inc/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.88, edgecolor='none'),
)

maybe_save(fig, output_dir / 'gt10_overlay_trackline_mangrove_relative_change_map.png', dpi=320)
plt.show()


In [ ]:
# Overlay map (Jamaica buffered extent): same layers, reduced sea area
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

# Buffer around Jamaica for map extent only (does not alter data)
JAMAICA_MAP_BUFFER_KM = 60

# Rebuild class raster (same logic as full map)
cls_rel = np.full(arr_before.shape, np.nan, dtype='float32')
cls_rel[paired_mask] = 0.0
cls_rel[eligible_mask & (rel_change < REL_DAMAGE_THRESHOLD)] = -1.0
cls_rel[eligible_mask & (rel_change > abs(REL_DAMAGE_THRESHOLD))] = 1.0

n_dec = int(np.nansum(cls_rel == -1.0))
n_mid = int(np.nansum(cls_rel == 0.0))
n_inc = int(np.nansum(cls_rel == 1.0))
n_tot = n_dec + n_mid + n_inc

cmap = ListedColormap(['#d73027', '#d9d9d9', '#1a9850'])
cmap.set_bad(color='white', alpha=1.0)
norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)

h, w = arr_before.shape
left, bottom, right, top = rasterio.transform.array_bounds(h, w, transform)

# Buffered Jamaica extent
jamaica_buffer_geom = unary_union(jamaica.geometry.tolist()).buffer(JAMAICA_MAP_BUFFER_KM * 1000)
xmin, ymin, xmax, ymax = jamaica_buffer_geom.bounds

fig, ax = plt.subplots(figsize=(12.5, 9.5), constrained_layout=True)
ax.imshow(
    cls_rel,
    cmap=cmap,
    norm=norm,
    extent=[left, right, bottom, top],
    origin='upper',
    interpolation='nearest',
)

# Same overlays
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw = track_windswath.copy()
    sw['RADII'] = pd.to_numeric(sw['RADII'], errors='coerce')
    threshold_colors = {34.0: '#9ecae1', 50.0: '#4292c6', 64.0: '#08519c'}
    for thr in sorted(sw['RADII'].dropna().unique()):
        subset = sw[sw['RADII'] == thr]
        color = threshold_colors.get(float(thr), '#6baed6')
        if len(subset) > 0:
            subset.boundary.plot(ax=ax, color=color, linewidth=1.0, alpha=0.9)

track_line.plot(ax=ax, color='black', linewidth=1.8, alpha=0.9)
jamaica.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.7)

# Clip map view to buffered Jamaica
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

ax.set_title(
    f'FN Mangroves NDVI Change + Melissa Track (Jamaica buffered view: {JAMAICA_MAP_BUFFER_KM} km)\n'
    f'Red: decrease >10% | Green: increase >10% | Grey: not substantial/low baseline',
    fontsize=11,
)
ax.set_xlabel('Easting (m, EPSG:3448)')
ax.set_ylabel('Northing (m, EPSG:3448)')
ax.set_aspect('equal')

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='NDVI decrease > 10%'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='none', label='Not substantial / low baseline'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='NDVI increase > 10%'),
    Line2D([0], [0], color='black', lw=1.8, label='Melissa track line'),
]
if len(track_windswath) > 0 and 'RADII' in track_windswath.columns:
    sw_r = pd.to_numeric(track_windswath['RADII'], errors='coerce')
    for thr, col in [(34.0, '#9ecae1'), (50.0, '#4292c6'), (64.0, '#08519c')]:
        if np.any(sw_r == thr):
            legend_handles.append(Line2D([0], [0], color=col, lw=1.2, label=f'Windswath {int(thr)} kt'))

ax.legend(handles=legend_handles, loc='upper right', frameon=True, framealpha=0.95, fontsize=9)

ax.text(
    0.01,
    0.01,
    f'n={n_tot:,}  dec={100*n_dec/max(n_tot,1):.1f}%  other={100*n_mid/max(n_tot,1):.1f}%  inc={100*n_inc/max(n_tot,1):.1f}%',
    transform=ax.transAxes,
    fontsize=9,
    ha='left',
    va='bottom',
    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.88, edgecolor='none'),
)

maybe_save(fig, output_dir / 'gt10_overlay_trackline_mangrove_relative_change_map_jamaica_buffered.png', dpi=320)
plt.show()


In [ ]:
# Optional table exports
if SAVE_OUTPUTS:
    step1.to_csv(output_dir / 'gt10_step1_inside_outside_swath.csv', index=False)
    step2.to_csv(output_dir / 'gt10_step2_distance_bins.csv', index=False)
    if len(step3_intensity) > 0:
        step3_intensity.to_csv(output_dir / 'gt10_step3_intensity_quantiles.csv', index=False)
    if len(step3_mslp) > 0:
        step3_mslp.to_csv(output_dir / 'gt10_step3_mslp_quantiles.csv', index=False)
    step4_quad.to_csv(output_dir / 'gt10_step4_quadrant_concentration.csv', index=False)
    step4_side.to_csv(output_dir / 'gt10_step4_side_concentration.csv', index=False)
    print('Saved CSV summaries to:', output_dir)
else:
    print('CSV export skipped (SAVE_OUTPUTS=False)')


## Notes
- This notebook targets only `>10%` relative NDVI decline areas, matching your request.
- Relative change is evaluated only for baseline `NDVI before >= 0.20`.
- Existing notebooks/files were not modified.

- Hurricane layers are clipped to a Jamaica-focused AOI (`TRACK_AOI_BUFFER_KM`) before analyses.


## Added at End: Continuous Distance Concentration of >10% Damage

This section uses **continuous distance to the Melissa track line** (not arbitrary thresholds) to show where `>10%` mangrove NDVI damage is most concentrated.


In [ ]:
# Continuous distance analysis (no arbitrary distance thresholds)
# Uses existing eligible_pixels + track_line built above.

from shapely.ops import unary_union

if 'line_union' not in globals() or line_union is None:
    line_union = unary_union(track_line.geometry.tolist())

dist_all = eligible_pixels.geometry.distance(line_union) / 1000.0

# Resolve damaged flag name robustly across notebook versions
if 'damaged_gt10pct' in eligible_pixels.columns:
    damage_flag = eligible_pixels['damaged_gt10pct']
elif 'is_damaged' in eligible_pixels.columns:
    damage_flag = eligible_pixels['is_damaged']
else:
    damage_flag = eligible_pixels['rel_change'] < REL_DAMAGE_THRESHOLD

dist_dmg = dist_all[damage_flag]

# Quantiles of damaged distances
q_levels = [0.10, 0.25, 0.50, 0.75, 0.90]
q_vals = np.quantile(dist_dmg, q_levels)

# Cumulative concentration distances
def cum_distance(sorted_vals, p):
    i = int(np.ceil(p * len(sorted_vals))) - 1
    i = max(0, min(i, len(sorted_vals) - 1))
    return float(sorted_vals[i])

d_sorted = np.sort(dist_dmg.to_numpy())

# Highest-density interval (HDI): tightest interval containing fraction p of damaged distances
# This is data-driven and avoids pre-set bands.
def hdi_interval(sorted_vals, p):
    n = len(sorted_vals)
    k = int(np.floor(p * n))
    if k < 1:
        return float(sorted_vals[0]), float(sorted_vals[-1])
    widths = sorted_vals[k:] - sorted_vals[:n-k]
    j = int(np.argmin(widths))
    return float(sorted_vals[j]), float(sorted_vals[j + k])

hdi50 = hdi_interval(d_sorted, 0.50)
hdi80 = hdi_interval(d_sorted, 0.80)

# Data-driven histogram width for identifying concentration peak
q1, q3 = np.quantile(dist_dmg, [0.25, 0.75])
iqr = q3 - q1
n = len(dist_dmg)
bw = 2 * iqr / (n ** (1/3)) if (iqr > 0 and n > 1) else max(0.5, float(np.std(dist_dmg)) / 10)
if (not np.isfinite(bw)) or (bw <= 0):
    bw = 0.5

mn, mx = float(np.min(dist_dmg)), float(np.max(dist_dmg))
nbins = int(np.ceil((mx - mn) / bw))
nbins = max(15, min(nbins, 200))
edges = np.linspace(mn, mx, nbins + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

hist_dmg, _ = np.histogram(dist_dmg, bins=edges)
hist_all, _ = np.histogram(dist_all, bins=edges)
rate = np.divide(hist_dmg, hist_all, out=np.full_like(hist_dmg, np.nan, dtype=float), where=hist_all > 0)

# Smooth lightly to stabilize the peak estimate
kernel = np.array([1, 2, 3, 2, 1], dtype=float)
kernel /= kernel.sum()
sm_dmg = np.convolve(hist_dmg, kernel, mode='same')
sm_rate = np.convolve(np.nan_to_num(rate, nan=0.0), kernel, mode='same')

peak_count_idx = int(np.nanargmax(sm_dmg))
peak_rate_idx = int(np.nanargmax(sm_rate))

summary_distance = pd.DataFrame([
    {'metric': 'Eligible mangrove pixels', 'value': int(len(eligible_pixels))},
    {'metric': 'Damaged mangrove pixels (>10% decline)', 'value': int(damage_flag.sum())},
    {'metric': 'Damaged share of eligible (%)', 'value': float(100 * damage_flag.mean())},
    {'metric': 'Damaged distance p10 (km)', 'value': float(q_vals[0])},
    {'metric': 'Damaged distance p25 (km)', 'value': float(q_vals[1])},
    {'metric': 'Damaged distance p50 (km)', 'value': float(q_vals[2])},
    {'metric': 'Damaged distance p75 (km)', 'value': float(q_vals[3])},
    {'metric': 'Damaged distance p90 (km)', 'value': float(q_vals[4])},
    {'metric': 'Distance containing 50% of damage (km)', 'value': float(cum_distance(d_sorted, 0.50))},
    {'metric': 'Distance containing 80% of damage (km)', 'value': float(cum_distance(d_sorted, 0.80))},
    {'metric': 'Distance containing 90% of damage (km)', 'value': float(cum_distance(d_sorted, 0.90))},
    {'metric': 'HDI 50% lower bound (km)', 'value': float(hdi50[0])},
    {'metric': 'HDI 50% upper bound (km)', 'value': float(hdi50[1])},
    {'metric': 'HDI 80% lower bound (km)', 'value': float(hdi80[0])},
    {'metric': 'HDI 80% upper bound (km)', 'value': float(hdi80[1])},
    {'metric': 'Peak distance (max damaged count, km)', 'value': float(centers[peak_count_idx])},
    {'metric': 'Peak distance (max damage rate, km)', 'value': float(centers[peak_rate_idx])},
])

summary_distance['value'] = summary_distance['value'].astype(float).round(3)
display(summary_distance)

# Simple visual check: damaged distance distribution + cumulative concentration
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

axes[0].hist(dist_dmg, bins=60, color='#d73027', alpha=0.85)
axes[0].axvline(float(centers[peak_count_idx]), color='black', linestyle='--', linewidth=1.4, label='Peak count distance')
axes[0].set_title('Damaged Mangrove Pixels: Distance to Track')
axes[0].set_xlabel('Distance to track (km)')
axes[0].set_ylabel('Damaged pixel count')
axes[0].legend(loc='upper right', frameon=True)

y = np.arange(1, len(d_sorted) + 1) / len(d_sorted)
axes[1].plot(d_sorted, y, color='#08519c', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle=':', linewidth=1)
axes[1].axhline(0.8, color='gray', linestyle=':', linewidth=1)
axes[1].axvline(cum_distance(d_sorted, 0.50), color='#238b45', linestyle='--', linewidth=1.4, label='50% distance')
axes[1].axvline(cum_distance(d_sorted, 0.80), color='#756bb1', linestyle='--', linewidth=1.4, label='80% distance')
axes[1].set_title('Cumulative Concentration of Damaged Pixels')
axes[1].set_xlabel('Distance to track (km)')
axes[1].set_ylabel('Cumulative share of damaged pixels')
axes[1].set_ylim(0, 1.02)
axes[1].legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()


## Added at End: Are All Mangroves Damaged Within 80%/90% Distances?

This checks, within the two damage-concentration distances, whether **all** FN mangrove pixels are `>10%` NDVI decline or whether non-damaged pixels are also present.

Notes:
- Distances are measured to the Melissa track line.
- Distances are derived from the damaged-pixel distribution (80% and 90% cumulative).
- Categories are on paired FN mangrove pixels: decrease >10%, increase >10%, and other/low-baseline.


In [ ]:
# Check for non-damaged mangroves inside the 80%/90% damaged-distance thresholds
# and visualize with FN mangrove boundaries.

from shapely.ops import unary_union
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

if 'line_union' not in globals() or line_union is None:
    line_union = unary_union(track_line.geometry.tolist())

# Build paired-pixel table (all paired mangrove pixels, not only baseline-eligible)
rows_p, cols_p = np.where(paired_mask)
xs_p, ys_p = rasterio.transform.xy(transform, rows_p, cols_p, offset='center')

rel_vals_p = rel_change[paired_mask]  # NaN where baseline is below REL_BASELINE_MIN
eligible_vals_p = eligible_mask[paired_mask]

is_dec = eligible_vals_p & (rel_vals_p < REL_DAMAGE_THRESHOLD)
is_inc = eligible_vals_p & (rel_vals_p > abs(REL_DAMAGE_THRESHOLD))

status = np.where(
    is_dec,
    'decrease_gt10',
    np.where(is_inc, 'increase_gt10', 'other_or_lowbaseline')
)

paired_pixels = gpd.GeoDataFrame(
    {
        'row': rows_p,
        'col': cols_p,
        'eligible_rel': eligible_vals_p,
        'rel_change': rel_vals_p,
        'is_damaged_gt10': is_dec,
        'status': status,
    },
    geometry=gpd.points_from_xy(xs_p, ys_p),
    crs=3448,
)

paired_pixels['distance_to_track_km'] = paired_pixels.geometry.distance(line_union) / 1000.0

# Derive 80% and 90% thresholds from the damaged-pixel distance distribution
if paired_pixels['is_damaged_gt10'].sum() == 0:
    raise ValueError('No damaged (>10% decline) mangrove pixels found.')

dmg_dist = paired_pixels.loc[paired_pixels['is_damaged_gt10'], 'distance_to_track_km'].to_numpy()
d80 = float(np.quantile(dmg_dist, 0.80))
d90 = float(np.quantile(dmg_dist, 0.90))

thresholds = [
    ('80% damaged distance', d80),
    ('90% damaged distance', d90),
]

rows = []
for label, thr in thresholds:
    sub = paired_pixels[paired_pixels['distance_to_track_km'] <= thr]
    n_total = int(len(sub))
    n_dec = int(sub['is_damaged_gt10'].sum())
    n_inc = int((sub['status'] == 'increase_gt10').sum())
    n_other = int((sub['status'] == 'other_or_lowbaseline').sum())
    n_non = n_total - n_dec

    rows.append({
        'threshold_label': label,
        'distance_km': thr,
        'paired_mangrove_pixels_within': n_total,
        'damaged_gt10_within': n_dec,
        'non_damaged_within': n_non,
        'increase_gt10_within': n_inc,
        'other_or_lowbaseline_within': n_other,
        'pct_damaged_of_within': 100.0 * n_dec / max(n_total, 1),
        'pct_non_damaged_of_within': 100.0 * n_non / max(n_total, 1),
        'all_within_are_damaged': (n_non == 0),
    })

threshold_summary = pd.DataFrame(rows)
threshold_summary['distance_km'] = threshold_summary['distance_km'].round(3)
for c in ['pct_damaged_of_within', 'pct_non_damaged_of_within']:
    threshold_summary[c] = threshold_summary[c].round(2)

display(threshold_summary)

# Explicit answer line
for _, r in threshold_summary.iterrows():
    if bool(r['all_within_are_damaged']):
        print(f"{r['threshold_label']} (<= {r['distance_km']} km): all paired mangrove pixels are damaged >10%.")
    else:
        print(
            f"{r['threshold_label']} (<= {r['distance_km']} km): "
            f"non-damaged pixels are present ({int(r['non_damaged_within']):,} of {int(r['paired_mangrove_pixels_within']):,})."
        )

# Map with FN mangrove boundaries and pixel categories within each threshold
MAX_POINTS_PER_PANEL = 90000
RANDOM_SEED = 42
JAMAICA_MAP_BUFFER_KM = 60

def stratified_sample(df, col, max_n, seed=42):
    if len(df) <= max_n:
        return df
    pieces = []
    total = len(df)
    for k, g in df.groupby(col):
        n_take = max(1, int(round(max_n * len(g) / total)))
        pieces.append(g.sample(min(len(g), n_take), random_state=seed))
    out = pd.concat(pieces, ignore_index=False)
    if len(out) > max_n:
        out = out.sample(max_n, random_state=seed)
    return out

fig, axes = plt.subplots(1, 2, figsize=(15, 7.2), constrained_layout=True)

cat_style = [
    ('decrease_gt10', '#d73027', 'Decrease > 10%'),
    ('increase_gt10', '#1a9850', 'Increase > 10%'),
    ('other_or_lowbaseline', '#bdbdbd', 'Other / low baseline'),
]

jamaica_buffer = unary_union(jamaica.geometry.tolist()).buffer(JAMAICA_MAP_BUFFER_KM * 1000)
xmin, ymin, xmax, ymax = jamaica_buffer.bounds

for ax, (label, thr) in zip(axes, thresholds):
    sub_full = paired_pixels[paired_pixels['distance_to_track_km'] <= thr].copy()
    sub_plot = stratified_sample(sub_full, 'status', MAX_POINTS_PER_PANEL, seed=RANDOM_SEED)

    # FN mangrove boundaries for context
    fn.boundary.plot(ax=ax, color='black', linewidth=0.35, alpha=0.9)

    # Track line and threshold-distance buffer boundary
    track_line.plot(ax=ax, color='black', linewidth=1.5, alpha=0.95)
    gpd.GeoSeries([line_union.buffer(thr * 1000.0)], crs=3448).boundary.plot(
        ax=ax, color='#3182bd', linewidth=1.2, alpha=0.9, linestyle='--'
    )

    # Pixel categories
    for cat, col, _lab in cat_style:
        ss = sub_plot[sub_plot['status'] == cat]
        if len(ss) > 0:
            ss.plot(ax=ax, markersize=0.8, color=col, alpha=0.65)

    n_total = len(sub_full)
    n_dec = int((sub_full['status'] == 'decrease_gt10').sum())
    n_non = n_total - n_dec

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Easting (m, EPSG:3448)')
    ax.set_ylabel('Northing (m, EPSG:3448)')
    ax.set_title(
        f'{label}: <= {thr:.2f} km\n'
        f'damaged={100*n_dec/max(n_total,1):.1f}% | non-damaged={100*n_non/max(n_total,1):.1f}%'
    )

legend_handles = [
    mpatches.Patch(facecolor='#d73027', edgecolor='none', label='Decrease > 10%'),
    mpatches.Patch(facecolor='#1a9850', edgecolor='none', label='Increase > 10%'),
    mpatches.Patch(facecolor='#bdbdbd', edgecolor='none', label='Other / low baseline'),
    Line2D([0], [0], color='black', lw=1.5, label='Melissa track line'),
    Line2D([0], [0], color='#3182bd', lw=1.2, linestyle='--', label='Distance threshold boundary'),
    Line2D([0], [0], color='black', lw=1.0, label='FN mangrove boundaries'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3, frameon=True, fontsize=9)

plt.show()
